In [1]:
import torch
from torch.utils.data import DataLoader
from torch import nn
import torch.optim as optim
import torchvision.models as models

import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'functions')))
from dataset import ChestXrayDataset
from train import train
from evaluation import plot_results ,eval_on_metrics
from gradcam import get_heatmap_for_resnet

In [2]:
IMAGE_PATH = "../archive/"
import glob

# Tüm alt klasörlerdeki jpg ve png dosyalarını alalım
image_paths = glob.glob(IMAGE_PATH + "**/images/*.[jp][pn]g", recursive=True)

print(f"Toplam {len(image_paths)} resim bulundu.")

Toplam 112120 resim bulundu.


In [3]:
TRAIN_PATH = '../data/AP_PA_Train.xlsx'
TEST_PATH = '../data/AP_PA_Test.xlsx'
VAL_PATH = '../data/AP_PA_Validation.xlsx'

In [4]:
num_classes = 2
EPOCHS = 30

In [5]:
model = models.resnet50(pretrained=True)

for param in model.parameters():
    param.requires_grad = False

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)

for param in model.fc.parameters():
    param.requires_grad = True

d:\anaconda\envs\ml\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\anaconda\envs\ml\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
model.parameters()

<generator object Module.parameters at 0x000001A54C3ED700>

In [8]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [9]:
train_dataset = ChestXrayDataset(TRAIN_PATH, image_paths,transform=transform)
val_dataset = ChestXrayDataset(TEST_PATH,image_paths, transform=transform)
test_dataset = ChestXrayDataset(VAL_PATH,image_paths, transform=transform)

In [10]:
print("Train size : ",len(train_dataset))
print("Validation size : ",len(val_dataset))
print("Test size : ",len(test_dataset))

Train size :  78566
Validation size :  16491
Test size :  17063


In [11]:
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True,num_workers=18, pin_memory=True)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=True,num_workers=18, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=True,num_workers=10)

In [12]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score

def objective(trial):
    # 🔁 Hiperparametreleri seç
    lr = trial.suggest_loguniform('lr', 1e-5, 1e-2)
    weight_decay = trial.suggest_loguniform('weight_decay', 1e-6, 1e-2)
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'AdamW', 'SGD'])
    
    model.to(device)

    # 🔁 Optimizasyon fonksiyonu
    if optimizer_name == 'Adam':
        optimizer = optim.Adam(model.fc.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == 'AdamW':
        optimizer = optim.AdamW(model.fc.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = optim.SGD(model.fc.parameters(), lr=lr, weight_decay=weight_decay, momentum=0.9)

    criterion = nn.CrossEntropyLoss()

    # 🔼 Eğitim
    num_epochs = 3
    for epoch in range(num_epochs):
        model.train()
        for inputs, labels in tqdm(train_dataloader, leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

    # 🧪 Doğrulama
    model.eval()
    preds = []
    targets = []
    with torch.no_grad():
        for inputs, labels in val_dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            pred = torch.argmax(outputs, dim=1)
            preds.extend(pred.cpu().numpy())
            targets.extend(labels.cpu().numpy())

    # 🎯 Başarı metriği
    acc = accuracy_score(targets, preds)
    return acc


In [13]:
import optuna

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

print("🔍 En iyi parametreler:", study.best_params)
print("✅ En yüksek doğruluk:", study.best_value)

d:\anaconda\envs\ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-07-28 23:25:50,884] A new study created in memory with name: no-name-5f9bf127-aa76-43b3-a25e-1e798eab7107
C:\Users\Furkan\AppData\Local\Temp\ipykernel_9404\3583492758.py:6: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform('lr', 1e-5, 1e-2)
C:\Users\Furkan\AppData\Local\Temp\ipykernel_9404\3583492758.py:7: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform('weight

🔍 En iyi parametreler: {'lr': 5.824225838297746e-05, 'weight_decay': 0.00010373923178610046, 'optimizer': 'Adam'}
✅ En yüksek doğruluk: 0.9921169122551695


In [ ]:
plot_results(train_losses, train_accuracies, val_losses, val_accuracies)

In [ ]:
eval_on_metrics(model, test_dataloader)

In [ ]:
test_dataset[0]

In [ ]:
model = models.resnet50(pretrained=True) 

# Son katmanı 2 sınıflı yap
model.fc = nn.Linear(model.fc.in_features, 2)

checkpoint_path = 'models/best_model.pth'
state_dict = torch.load(checkpoint_path)

# 3. state_dict'i modele yükle
model.load_state_dict(state_dict)

# 4. Modele eval modunu ver (inference için)
model.eval()

In [ ]:
test_image_path = test_dataset.get_path(0)

In [ ]:
get_heatmap_for_resnet(model,test_image_path)